In [7]:
import pandas as pd
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error, r2_score
import warnings

warnings.filterwarnings("ignore")

df = pd.read_csv('/ssd1/muntasir/Desktop/AutoTimes/dataset/panel/panel_gvkey_filtered.csv')


In [8]:
df

,DATE,gvkey,mean,std,capital_ratio,equity_invcap,debt_invcap,totdebt_invcap,at_turn,pay_turn,...,debt_capital,bm,CAPEI,evm,pe_exi,pe_inc,pe_op_basic,ps,ptb,actual
0,1990-03,1300.0,0.203125,0.009613,0.515901,0.489510,0.523297,0.353982,0.734494,0.632692,...,0.471616,0.696538,0.182221,0.009147,0.345252,0.331681,0.289909,0.034120,0.061408,0.2250
1,1990-06,1300.0,0.215000,0.018841,0.544170,0.461538,0.551971,0.315634,0.743199,0.740513,...,0.423581,0.665988,0.195695,0.005676,0.345027,0.331464,0.007462,0.032618,0.062676,0.2200
2,1990-09,1300.0,0.201875,0.023443,0.519435,0.486014,0.526882,0.407080,0.737758,0.734487,...,0.480349,0.680244,0.089551,0.019672,0.342997,0.329515,0.204373,0.006652,0.013099,0.1900
3,1990-12,1300.0,0.205000,0.011055,0.501767,0.503497,0.508961,0.522124,0.738847,0.732179,...,0.545852,0.963340,0.073743,0.019782,0.342852,0.329375,0.199526,0.002146,0.006338,0.2000
4,1991-03,1300.0,0.131111,0.028370,0.586572,0.419580,0.594982,0.525074,0.748640,0.630641,...,0.624454,1.000000,0.000000,0.005510,0.343401,0.329902,0.217798,0.006223,0.015634,0.1175
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3336,2020-12,10903.0,2.410429,0.091870,0.614035,0.360976,0.542773,0.639942,0.215385,0.404638,...,0.696825,0.128099,0.491253,0.470724,0.763672,0.763672,0.467046,0.323632,0.480405,2.5200
3337,2021-03,10903.0,4.395105,0.199896,0.591707,0.376829,0.522124,0.580175,0.240000,0.464174,...,0.665079,0.095041,0.491125,0.686808,0.797032,0.797032,0.540802,0.339766,0.513626,5.3100
3338,2021-06,10903.0,4.428100,0.140454,0.596491,0.375610,0.528024,0.602041,0.203077,0.415369,...,0.674603,0.082645,0.491383,0.573874,0.792184,0.792184,0.530435,0.361594,0.551259,4.7000
3339,2021-09,10903.0,4.386250,0.082156,0.599681,0.374390,0.530973,0.596210,0.215385,0.477674,...,0.673016,0.073003,0.491169,0.777298,0.825077,0.825077,0.602907,0.335337,0.508824,4.5200


In [9]:

# -----------------------
# Cutoffs
# -----------------------
CUTOFF_TRAIN = pd.Timestamp('2012-01-01')
CUTOFF_VAL   = pd.Timestamp('2015-01-01')

# -----------------------
# Prep (ensure datetime)
# -----------------------
df['DATE'] = pd.to_datetime(df['DATE'])
df = df.sort_values(['gvkey', 'DATE'])

# -----------------------
# Function: Fit ARIMA per gvkey
# -----------------------
results = []

for gvkey, group in df.groupby('gvkey'):
    group = group.sort_values('DATE')

    train_data = group[group['DATE'] < CUTOFF_TRAIN]
    val_data   = group[(group['DATE'] >= CUTOFF_TRAIN) & (group['DATE'] < CUTOFF_VAL)]

    # Skip if not enough training data
    if len(train_data) < 10 or len(val_data) == 0:
        continue

    y_train = train_data['actual'].values
    y_val = val_data['actual'].values
    dates_val = val_data['DATE'].values

    try:
        # Fit ARIMA model (p,d,q) = (1,1,0) is a reasonable start
        model = ARIMA(y_train, order=(1,1,0))
        model_fit = model.fit()

        # Forecast for length of validation
        forecast = model_fit.forecast(steps=len(y_val))

        mse = mean_squared_error(y_val, forecast)
        r2 = r2_score(y_val, forecast)
        results.append({
            'gvkey': gvkey,
            'mse': mse,
            'r2': r2,
            'n_train': len(y_train),
            'n_val': len(y_val)
        })
    except Exception as e:
        print(f"❌ ARIMA failed for gvkey={gvkey}: {e}")
        continue

# -----------------------
# Results Summary
# -----------------------
results_df = pd.DataFrame(results)
print("📊 Avg MSE:", results_df['mse'].mean())
print(results_df.sort_values('mse').head())

📊 Avg MSE: 0.18305589178853912
       gvkey       mse        r2  n_train  n_val
24   64768.0  0.000555 -0.552614       58     12
25  117768.0  0.000581 -1.165513       45     12
26  157855.0  0.000754 -0.913211       30     12
22   20779.0  0.005586 -6.392718       86     12
15    7257.0  0.008390 -1.067778       88     12


In [10]:
# -----------------------
# Containers for aggregation
# -----------------------
results = []
all_y_val = []
all_y_pred = []

for gvkey, group in df.groupby('gvkey'):
    group = group.sort_values('DATE')

    train_data = group[group['DATE'] < CUTOFF_TRAIN]
    val_data   = group[(group['DATE'] >= CUTOFF_TRAIN) & (group['DATE'] < CUTOFF_VAL)]

    if len(train_data) < 10 or len(val_data) == 0:
        continue

    y_train = train_data['actual'].values
    y_val = val_data['actual'].values

    try:
        model = ARIMA(y_train, order=(1,1,0))
        model_fit = model.fit()

        forecast = model_fit.forecast(steps=len(y_val))

        mse = mean_squared_error(y_val, forecast)
        r2 = r2_score(y_val, forecast)

        results.append({
            'gvkey': gvkey,
            'mse': mse,
            'r2': r2,
            'n_train': len(y_train),
            'n_val': len(y_val)
        })

        # For global aggregation
        all_y_val.extend(y_val)
        all_y_pred.extend(forecast)

    except Exception as e:
        print(f"❌ ARIMA failed for gvkey={gvkey}: {e}")
        continue

In [11]:
# -----------------------
# Results Summary
# -----------------------
results_df = pd.DataFrame(results)

# Global aggregated R² and MSE
global_mse = mean_squared_error(all_y_val, all_y_pred)
global_r2  = r2_score(all_y_val, all_y_pred)

print("\n📈 Aggregated Validation Results Across All GVKEYs")
print("---------------------------------------------------")
print(f"Global MSE : {global_mse:.4f}")
print(f"Global R²  : {global_r2:.4f}")
print("\n📊 Per-GVKEY Summary")
print(results_df.describe())


📈 Aggregated Validation Results Across All GVKEYs
---------------------------------------------------
Global MSE : 0.1831
Global R²  : 0.7806

📊 Per-GVKEY Summary
               gvkey        mse         r2    n_train  n_val
count      28.000000  28.000000  28.000000  28.000000   28.0
mean    25646.714286   0.183056  -2.151928  79.321429   12.0
std     47910.270999   0.312107   2.321863  19.710974    0.0
min      1300.000000   0.000555 -10.165075  16.000000   12.0
25%      2930.250000   0.015894  -2.425954  87.000000   12.0
50%      6710.000000   0.059780  -1.455161  88.000000   12.0
75%     10992.000000   0.205461  -0.930534  88.000000   12.0
max    179534.000000   1.217083   0.037410  88.000000   12.0


In [13]:
import pandas as pd
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error, r2_score
import warnings

warnings.filterwarnings("ignore")

df = pd.read_csv('/ssd1/muntasir/Desktop/AutoTimes/dataset/panel/panel_gvkey_filtered.csv')

# -----------------------
# Cutoffs
# -----------------------
CUTOFF_TRAIN = pd.Timestamp('2012-01-01')
CUTOFF_VAL   = pd.Timestamp('2015-01-01')  

# -----------------------
# Prep (ensure datetime)
# -----------------------
df['DATE'] = pd.to_datetime(df['DATE'])
df = df.sort_values(['gvkey', 'DATE'])

# -----------------------
# Containers for aggregation
# -----------------------
results = []
all_y_test = []
all_y_pred = []

for gvkey, group in df.groupby('gvkey'):
    group = group.sort_values('DATE')

    train_data = group[group['DATE'] < CUTOFF_VAL]
    test_data  = group[group['DATE'] >= CUTOFF_VAL]

    if len(train_data) < 10 or len(test_data) == 0:
        continue

    y_train = train_data['actual'].values
    y_test  = test_data['actual'].values

    try:
        model = ARIMA(y_train, order=(1, 1, 0))
        model_fit = model.fit()
        forecast = model_fit.forecast(steps=len(y_test))

        mse = mean_squared_error(y_test, forecast)
        r2 = r2_score(y_test, forecast)

        results.append({
            'gvkey': gvkey,
            'mse': mse,
            'r2': r2,
            'n_train': len(y_train),
            'n_test': len(y_test)
        })

        all_y_test.extend(y_test)
        all_y_pred.extend(forecast)

    except Exception as e:
        print(f"❌ ARIMA failed for gvkey={gvkey}: {e}")
        continue

# -----------------------
# Results Summary
# -----------------------
results_df = pd.DataFrame(results)

# Global aggregated R² and MSE on test set
global_mse = mean_squared_error(all_y_test, all_y_pred)
global_r2  = r2_score(all_y_test, all_y_pred)

print("\n📈 Aggregated Test Set Results Across All GVKEYs")
print("---------------------------------------------------")
print(f"Global MSE : {global_mse:.4f}")
print(f"Global R²  : {global_r2:.4f}")
print("\n📊 Per-GVKEY Summary")
print(results_df.describe())


📈 Aggregated Test Set Results Across All GVKEYs
---------------------------------------------------
Global MSE : 1.3081
Global R²  : 0.0929

📊 Per-GVKEY Summary
               gvkey        mse         r2     n_train  n_test
count      28.000000  28.000000  28.000000   28.000000    28.0
mean    25646.714286   1.308124  -1.564762   91.321429    28.0
std     47910.270999   2.525805   1.445719   19.710974     0.0
min      1300.000000   0.010896  -5.714658   28.000000    28.0
25%      2930.250000   0.168110  -2.202525   99.000000    28.0
50%      6710.000000   0.387719  -1.227422  100.000000    28.0
75%     10992.000000   1.481040  -0.463182  100.000000    28.0
max    179534.000000  12.792884  -0.019959  100.000000    28.0


In [14]:
results_df

,gvkey,mse,r2,n_train,n_test
0,1300.0,0.205053,-2.457081,100,28
1,1447.0,0.475286,-0.462013,99,28
2,1602.0,2.061208,-4.493057,100,28
3,1690.0,0.179245,-0.048952,100,28
4,2136.0,0.156913,-5.714658,100,28
5,2285.0,12.792884,-0.275950,100,28
6,2817.0,0.959117,-0.463572,100,28
7,2968.0,1.864766,-1.251145,100,28
8,2991.0,1.600054,-0.909880,100,28
9,3144.0,0.010896,-0.511907,100,28
